# Clase 06: Búsqueda Semántica sobre el Corpus de Report Generator

**Laboratorio en Google Colab - Corte 2 - Equipo Juliofi**

Previamente, el prototipo clasificaba la petición y consultaba una fuente estructurada (`consultar_fuente_datos`). Hoy sumamos la capacidad de buscar, en un corpus de texto, el pasaje que podría orientar una consulta. Todavía no generamos respuestas con esos pasajes.

**Antes:** texto -> clasificación -> validación -> herramienta -> borrador.

**Ahora:** consulta -> embedding -> similitud coseno -> ranking top-k -> lectura de lo recuperado para decidir si alcanza.

> *NOTA: El corpus es ficticio y lo escribimos a partir de las plantillas y reglas del prototipo (C04). No hay documentos reales de ninguna empresa.*

## 1. Preparación
Instalamos la biblioteca para generar embeddings. En una sesión nueva de Colab se ejecuta esta celda una vez.

In [ ]:
%pip -q install sentence-transformers

## 2. Nuestro Corpus

Seis pasajes cortos sobre cómo funciona Report Generator. Cada uno conserva identificador, texto, procedencia y fecha de consulta. Pregunta al grupo: **¿cuál de estos pasajes serviría para quien pide "el documento para la gerencia"?**

In [ ]:
from datetime import date

FECHA_CONSULTA = date.today().isoformat()
PROCEDENCIA = "Plantillas y reglas del prototipo Report Generator (Clase 04), texto ficticio"

corpus = [
    {"id": "P1", "texto": "El informe ejecutivo resume los indicadores clave para la dirección. Incluye contexto y periodo, indicadores y una sección de datos ausentes y riesgos."},
    {"id": "P2", "texto": "El informe técnico describe métricas del sistema, como uso de CPU, latencia p95 y errores 5xx, sobre un periodo observado."},
    {"id": "P3", "texto": "El resumen periódico es un informe recurrente, semanal o mensual, sobre el periodo que cubre la fuente de datos."},
    {"id": "P4", "texto": "El periodo cubierto y la última actualización de una fuente se consultan en el catálogo mediante su código DS-0000."},
    {"id": "P5", "texto": "Si una métrica está incompleta en la fuente, el borrador la marca como DATO INCOMPLETO y no la presenta como verificada."},
    {"id": "P6", "texto": "Publicar o enviar un informe no hace parte del servicio: el prototipo solo redacta borradores que requieren revisión humana."},
]
for pasaje in corpus:
    pasaje["fuente"] = PROCEDENCIA
    pasaje["fecha_consulta"] = FECHA_CONSULTA
    print(f"{pasaje['id']}: {pasaje['texto']}")

P1: El informe ejecutivo resume los indicadores clave para la dirección. Incluye contexto y periodo, indicadores y una sección de datos ausentes y riesgos.
P2: El informe técnico describe métricas del sistema, como uso de CPU, latencia p95 y errores 5xx, sobre un periodo observado.
P3: El resumen periódico es un informe recurrente, semanal o mensual, sobre el periodo que cubre la fuente de datos.
P4: El periodo cubierto y la última actualización de una fuente se consultan en el catálogo mediante su código DS-0000.
P5: Si una métrica está incompleta en la fuente, el borrador la marca como DATO INCOMPLETO y no la presenta como verificada.
P6: Publicar o enviar un informe no hace parte del servicio: el prototipo solo redacta borradores que requieren revisión humana.


## 3. Convertimos el corpus en vectores
Usamos un [modelo multilingüe de Sentence Transformers](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). `normalize_embeddings=True` deja los vectores con longitud unitaria; así, su producto punto corresponde a la similitud coseno. Los seis vectores se calculan **una sola vez**, antes de escribir la pregunta.

Mostramos la forma de la matriz y solo los primeros valores de un vector. Los números no se interpretan uno por uno como palabras o reglas.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

NOMBRE_MODELO = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
modelo = SentenceTransformer(NOMBRE_MODELO)
textos = [pasaje["texto"] for pasaje in corpus]
vectores_corpus = modelo.encode(textos, convert_to_numpy=True, normalize_embeddings=True)

print("Modelo:", NOMBRE_MODELO)
print("Forma de la matriz (pasajes, dimensiones):", vectores_corpus.shape)
print("Primeros 8 valores de P1:", np.round(vectores_corpus[0, :8], 3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Forma de la matriz (pasajes, dimensiones): (6, 384)
Primeros 8 valores de P1: [-0.018  0.004 -0.012  0.037  0.101  0.074 -0.034  0.041]


## 4. Una Búsqueda Literal Breve

La palabra `gerencia` aparecerá en nuestra pregunta, pero no está en el corpus (P1 dice "dirección"). Tampoco aparece el código `DS-1001`: P4 solo habla de `DS-0000`. Una coincidencia literal no encuentra nada en ninguno de los dos casos.

In [ ]:
for termino in ["gerencia", "DS-1001"]:
    coincidencias = [p["id"] for p in corpus if termino.casefold() in p["texto"].casefold()]
    print(f"Pasajes que contienen literalmente '{termino}': {coincidencias}")

Pasajes que contienen literalmente 'gerencia': []
Pasajes que contienen literalmente 'DS-1001': []


## 5. Escribe aquí la pregunta del usuario
**Esta es la celda donde ingresamos el texto del usuario.** En Colab se puede editar el campo `CONSULTA` del formulario o cambiar directamente el texto entre comillas y ejecutar de nuevo. Predicción antes de ejecutar: ¿esperamos ver P1 entre los primeros resultados?

> Predicción: esperamos ver P1 entre los 3 primeros.

In [ ]:
CONSULTA = "Necesito el documento para la gerencia con las cifras más importantes del negocio" #@param {type:"string"}
K = 3 #@param {type:"integer"}
print("Consulta recibida:", CONSULTA)

Consulta recibida: Necesito el documento para la gerencia con las cifras más importantes del negocio


## 6. Buscamos los pasajes más cercanos
Convertimos la consulta con el **mismo modelo**, calculamos la similitud coseno y ordenamos los resultados. `K` indica cuántos pasajes mostramos. El puntaje permite ordenar; no representa un porcentaje de certeza.

In [ ]:
def buscar(consulta, k=3):
    if not consulta.strip():
        raise ValueError("Escribe una consulta antes de buscar.")
    if not 1 <= k <= len(corpus):
        raise ValueError(f"K debe estar entre 1 y {len(corpus)}.")

    vector_consulta = modelo.encode(consulta, convert_to_numpy=True, normalize_embeddings=True)
    puntajes = vectores_corpus @ vector_consulta
    indices = np.argsort(-puntajes)[:k]
    return [(corpus[int(i)]["id"], float(puntajes[i]), corpus[int(i)]["texto"]) for i in indices]

resultados = buscar(CONSULTA, K)
for puesto, (identificador, similitud, texto) in enumerate(resultados, 1):
    print(f"{puesto}. {identificador} | similitud={similitud:.3f} | {texto}")

assert all(resultados[i][1] >= resultados[i + 1][1] for i in range(len(resultados) - 1))

1. P1 | similitud=0.566 | El informe ejecutivo resume los indicadores clave para la dirección. Incluye contexto y periodo, indicadores y una sección de datos ausentes y riesgos.
2. P3 | similitud=0.336 | El resumen periódico es un informe recurrente, semanal o mensual, sobre el periodo que cubre la fuente de datos.
3. P2 | similitud=0.278 | El informe técnico describe métricas del sistema, como uso de CPU, latencia p95 y errores 5xx, sobre un periodo observado.


## 7. La prueba importante: resultado cercano, información insuficiente
Ejecutamos dos preguntas que el corpus **no puede responder por completo**. Debemos leer el texto recuperado antes de decidir qué decir al usuario. No fijamos un umbral automático con seis ejemplos: un primer lugar siempre existe, incluso si falta la respuesta.

In [ ]:
for pregunta in ["¿Cómo calculo la tasa de retención de clientes?", "¿Cuál es la última actualización de DS-1001?"]:
    print("\nPregunta:", pregunta)
    for identificador, similitud, texto in buscar(pregunta, k=2):
        print(f"  {identificador} | {similitud:.3f} | {texto}")


Pregunta: ¿Cómo calculo la tasa de retención de clientes?
  P3 | 0.345 | El resumen periódico es un informe recurrente, semanal o mensual, sobre el periodo que cubre la fuente de datos.
  P2 | 0.327 | El informe técnico describe métricas del sistema, como uso de CPU, latencia p95 y errores 5xx, sobre un periodo observado.

Pregunta: ¿Cuál es la última actualización de DS-1001?
  P4 | 0.609 | El periodo cubierto y la última actualización de una fuente se consultan en el catálogo mediante su código DS-0000.
  P2 | 0.298 | El informe técnico describe métricas del sistema, como uso de CPU, latencia p95 y errores 5xx, sobre un periodo observado.


## 8. Batería de Pruebas con Expectativa Escrita

Escribimos el pasaje esperado antes de ejecutar. `None` significa "ningún pasaje suficiente". El chequeo automático solo dice si el esperado quedó entre los primeros K; no dice si el pasaje responde la pregunta. Eso lo decide la revisión humana leyendo el texto.

In [ ]:
PRUEBAS = [
    ("Necesito el documento para la gerencia con las cifras más importantes del negocio", "P1", "paráfrasis"),
    ("¿Qué métricas trae el informe con errores 5xx?", "P2", "coincidencia exacta"),
    ("¿Cada cuánto sale el informe?", "P3", "pregunta incompleta"),
    ("¿Cuál es la última actualización de DS-1001?", "P4", "el pasaje explica dónde consultar, no trae el dato"),
    ("¿Cómo calculo la tasa de retención de clientes?", None, "fuera de cobertura: el prototipo no calcula valores"),
]

EVALUACION = []
for i, (pregunta, esperado, tipo) in enumerate(PRUEBAS, 1):
    top = buscar(pregunta, K)
    ids = [r[0] for r in top]
    fila = {"caso": i, "tipo": tipo, "pregunta": pregunta, "esperado": esperado,
            "top_k": [(r[0], round(r[1], 3)) for r in top],
            "esperado_en_top_k": (esperado in ids) if esperado else None,
            "revision_humana": "PENDIENTE"}
    EVALUACION.append(fila)
    print(f"{i}. [{tipo}] esperado={esperado} | top-{K}={fila['top_k']} | en_top_k={fila['esperado_en_top_k']}")

CONFIG = {"modelo": NOMBRE_MODELO, "medida": "similitud coseno (vectores normalizados)", "k": K,
          "corpus": f"{len(corpus)} pasajes ficticios", "fecha_consulta": FECHA_CONSULTA}
print("\nConfiguración:", CONFIG)

1. [paráfrasis] esperado=P1 | top-3=[('P1', 0.566), ('P3', 0.336), ('P2', 0.278)] | en_top_k=True
2. [coincidencia exacta] esperado=P2 | top-3=[('P2', 0.729), ('P1', 0.551), ('P4', 0.394)] | en_top_k=True
3. [pregunta incompleta] esperado=P3 | top-3=[('P3', 0.457), ('P1', 0.385), ('P4', 0.315)] | en_top_k=True
4. [el pasaje explica dónde consultar, no trae el dato] esperado=P4 | top-3=[('P4', 0.609), ('P2', 0.298), ('P1', 0.231)] | en_top_k=True
5. [fuera de cobertura: el prototipo no calcula valores] esperado=None | top-3=[('P3', 0.345), ('P2', 0.327), ('P1', 0.252)] | en_top_k=None

Configuración: {'modelo': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'medida': 'similitud coseno (vectores normalizados)', 'k': 3, 'corpus': '6 pasajes ficticios', 'fecha_consulta': '2026-10-06'}


### Resultados de la batería (K = 3)

| # | Tipo | Esperado | Top 3 (puntaje) | ¿En top 3? | Lectura humana |
| - | ---- | -------- | --------------- | :--------: | -------------- |
| 1 | Paráfrasis | P1 | P1 0.566, P3 0.336, P2 0.278 | Sí | P1 sirve: dice "dirección" donde la pregunta dice "gerencia" |
| 2 | Coincidencia exacta | P2 | P2 0.729, P1 0.551, P4 0.394 | Sí | P2 lista los errores 5xx; P1 queda cerca (0.551) sin responder |
| 3 | Pregunta incompleta | P3 | P3 0.457, P1 0.385, P4 0.315 | Sí | P3 da la periodicidad en general (semanal o mensual), no la de un informe concreto |
| 4 | Dato que vive en la fuente | P4 | P4 0.609, P2 0.298, P1 0.231 | Sí | P4 explica dónde consultar, pero no trae la fecha; hace falta la herramienta |
| 5 | Fuera de cobertura | Ninguno | P3 0.345, P2 0.327, P1 0.252 | N/A | Ningún pasaje explica cómo calcular la tasa de retención |

### Preguntas para Cerrar (Adaptadas)

1. ¿Qué pasaje quedó primero ante la pregunta por "el documento para la gerencia"? ¿Coincidieron las palabras o el significado?
2. ¿Algún pasaje explica cómo calcular la tasa de retención? ¿Qué pasaje apareció igual, y por qué?
3. P4 explica dónde consultar la última actualización de una fuente. ¿Contiene la fecha de `DS-1001`? En la Clase 04 ese dato lo daba `consultar_fuente_datos`; aquí solo buscamos texto.

### Respuestas

1. **P1 quedó primero (0.566).** Coincidió el significado, no las palabras: la pregunta dice "gerencia" y "cifras más importantes", y P1 dice "dirección" e "indicadores clave". La búsqueda literal de `gerencia` no encontró nada.
2. **No.** Ningún pasaje explica cómo calcular la tasa de retención. Aun así aparecieron P3 (0.345) y P2 (0.327), porque siempre hay un "más cercano"; probablemente salen por compartir temática de informes y periodos. Por eso hay que leer el texto antes de responder.
3. **No.** P4 solo explica dónde consultar la última actualización de una fuente; la fecha de `DS-1001` sigue viviendo en el catálogo y la entrega `consultar_fuente_datos`. P4 salió primero (0.609) por el significado de "última actualización", no por el código, que ni aparece en el corpus.

### Lo Que Queda Pendiente

Este corpus son seis pasajes escritos por nosotras. Cuando tengamos documentos reales, hará falta fragmentarlos, conservar metadatos e indexarlos.

## Siguiente paso en el proyecto
Sustituyan `corpus` por unos pocos pasajes **autorizados y verificables** de su proyecto; conserven identificadores, procedencia y fecha de consulta. Ejecuten una paráfrasis, una coincidencia exacta y una pregunta fuera de cobertura. Registren en la bitácora el modelo, `K`, los pasajes recuperados y un fallo observado. La preparación e indexación de documentos completos corresponde a la Clase 07.

Documentación: [Sentence Transformers — Semantic Search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html).